In [ ]:
import pandas as pd
df=pd.read_excel("/aug_train.xlsx")
df.head()


,enrollee_id,city,city_development_index,gender,relevent_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,target
0,8949,city_103,0.920,Male,Has relevent experience,no_enrollment,Graduate,STEM,>20,NaN,NaN,1,36,1
1,29725,city_40,0.776,Male,No relevent experience,no_enrollment,Graduate,STEM,15,50-99,Pvt Ltd,>4,47,0
2,11561,city_21,0.624,NaN,No relevent experience,Full time course,Graduate,STEM,5,NaN,NaN,never,83,0
3,33241,city_115,0.789,NaN,No relevent experience,NaN,Graduate,Business Degree,<1,NaN,Pvt Ltd,never,52,1
4,666,city_162,0.767,Male,Has relevent experience,no_enrollment,Masters,STEM,>20,50-99,Funded Startup,4,8,0


In [ ]:
df.shape



(19158, 14)

In [ ]:
df.isnull().sum()

,0
enrollee_id,0
city,0
city_development_index,0
gender,4508
relevent_experience,0
enrolled_university,386
education_level,460
major_discipline,2813
experience,65
company_size,5938


In [ ]:
df["target"].value_counts()

,count
target,
0,14381
1,4777


In [ ]:
tot_yes=(df['target']==1).sum()
tot=len(df)
perc=(tot_yes/tot)*100
print("% of yes:",round(perc,2))

% of yes: 24.93


In [ ]:
miss=df.isnull().sum()
print(miss[miss>0].sort_values(ascending=False))

company_type           6140
company_size           5938
gender                 4508
major_discipline       2813
education_level         460
last_new_job            423
enrolled_university     386
experience               65
dtype: int64


In [ ]:
df_old=df.copy()
mis_col=["company_type","company_size","gender","major_discipline","education_level","last_new_job","enrolled_university","experience"]
df_old[mis_col]=df_old[mis_col].fillna("UNKNOWN")

In [ ]:
print(df_old["experience"].dtype)

object


In [ ]:
print(df_old["experience"].unique())

['>20' 15 5 '<1' 11 13 7 17 2 16 1 4 10 14 18 19 12 3 6 9 8 20 'UNKNOWN']


In [ ]:
print((df_old['experience']=="UNKNOWN").sum())

65


In [ ]:
print(df_old['experience'].isnull().sum())

0


predictive model


In [ ]:
x=df_old.drop(columns=["target","enrollee_id"])
y=df_old["target"]
print(x.shape)
print(y.shape)

(19158, 12)
(19158,)


In [ ]:
x_newtype=pd.get_dummies(x,drop_first=True,dtype=int)
print(x_newtype.shape)
print(x.head())

(19158, 184)
       city  city_development_index   gender      relevent_experience  \
0  city_103                   0.920     Male  Has relevent experience   
1   city_40                   0.776     Male   No relevent experience   
2   city_21                   0.624  UNKNOWN   No relevent experience   
3  city_115                   0.789  UNKNOWN   No relevent experience   
4  city_162                   0.767     Male  Has relevent experience   

  enrolled_university education_level major_discipline experience  \
0       no_enrollment        Graduate             STEM        >20   
1       no_enrollment        Graduate             STEM         15   
2    Full time course        Graduate             STEM          5   
3             UNKNOWN        Graduate  Business Degree         <1   
4       no_enrollment         Masters             STEM        >20   

  company_size    company_type last_new_job  training_hours  
0      UNKNOWN         UNKNOWN            1              36  
1        

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(
    x_newtype,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(15326, 184)
(3832, 184)
(15326,)
(3832,)


In [ ]:
from sklearn.linear_model import LogisticRegression
model=LogisticRegression(max_iter=1000)
model.fit(x_train,y_train)

LogisticRegression(max_iter=1000)

In [ ]:
pred=model.predict(x_test)
print("actual:")
print(y_test.head(10).values)
print("predicted:")
print(pred[:10])

actual:
[0 1 0 1 1 0 0 0 1 0]
predicted:
[0 0 0 0 1 1 0 0 0 0]


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test,pred)

print("Model Accuracy:", round(accuracy * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(y_test, pred))

Model Accuracy: 78.13 %

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.91      0.86      2877
           1       0.59      0.39      0.47       955

    accuracy                           0.78      3832
   macro avg       0.71      0.65      0.66      3832
weighted avg       0.76      0.78      0.76      3832

